# Anwiaso East Forest Reserve — Data Preparation
## Google Earth Engine Python API · Google Colab

Prepares all inputs needed by the RF and DL Colab classification notebooks.

**Outputs saved to Google Drive `Forest_LULC_Classification/`:**
- `s1_s2_features_2026.tif` — 26-band S1+S2 feature stack (set via `CONFIG['features_file']`)
- `training_samples.csv` — labelled sample points with extracted band values (set via `CONFIG['samples_file']`)
- `class_mapping.json` — integer-id to class-name lookup consumed by both Colab classification notebooks

**Run order:** execute cells top to bottom. Edit `CONFIG` before running.

> **File names must match** `CONFIG['features_file']` and `CONFIG['samples_file']`
> in `forest_classification_using_rf_and_colab.ipynb` and
> `forest_classification_dl_and_colab.ipynb`.

In [ ]:
# Run this cell first
!pip install earthengine-api geemap geopandas fiona -q


In [1]:
import ee, io, json, os
import geopandas as gpd
import pandas as pd
import geemap
from pathlib import Path
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# NEW SESSION  -> uncomment ee.Authenticate(), run, paste code, comment out.
# SAME SESSION -> leave ee.Authenticate() commented; just run ee.Initialize().
ee.Authenticate(auth_mode='notebook')   # <- uncomment for a new session only
ee.Initialize(project='data-pipeline-1-464218')
print('GEE initialised')

# Drive REST API -- reuses the OAuth token saved by ee.Authenticate()
# so no extra browser popup is needed.
_creds_path = os.path.expanduser('~/.config/earthengine/credentials')
with open(_creds_path) as _f:
    _cd = json.load(_f)
_drive_creds = Credentials(
    token=None, refresh_token=_cd['refresh_token'],
    token_uri='https://oauth2.googleapis.com/token',
    client_id=_cd['client_id'], client_secret=_cd['client_secret'],
    scopes=['https://www.googleapis.com/auth/drive']
)
_drive_creds.refresh(Request())
_drive_svc = build('drive', 'v3', credentials=_drive_creds)

# Auto-detects Colab, Lightning AI, or local machine
_BASE = (
    Path('/content') if Path('/content').exists() else
    Path('/teamspace/studios/this_studio') if Path('/teamspace').exists() else
    Path.home()
)
LOCAL_DIR = _BASE / 'Forest_LULC_Classification'
LOCAL_DIR.mkdir(exist_ok=True)


def _folder_id():
    res = _drive_svc.files().list(
        q=f"name='{CONFIG['drive_folder']}' and "
          f"mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields='files(id)'
    ).execute()
    if not res['files']:
        raise FileNotFoundError(f"Drive folder '{CONFIG['drive_folder']}' not found.")
    return res['files'][0]['id']


def drive_download(filename):
    dest = LOCAL_DIR / filename
    if dest.exists():
        print(f'   Already local: {filename}')
        return str(dest)
    fid = _folder_id()
    res = _drive_svc.files().list(
        q=f"name='{filename}' and '{fid}' in parents and trashed=false",
        fields='files(id)'
    ).execute()
    if not res['files']:
        raise FileNotFoundError(
            f"'{filename}' not found in Drive/{CONFIG['drive_folder']}/"
        )
    req = _drive_svc.files().get_media(fileId=res['files'][0]['id'])
    with io.FileIO(str(dest), 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = dl.next_chunk()
    print(f'   Downloaded: {filename}')
    return str(dest)


print('Drive REST API ready')
print(f'Local scratch : {LOCAL_DIR}')


GEE Python API initialised.


In [2]:
# ============================================================
# CONFIG — edit all settings here
# ============================================================
# Key names match forest_classification_using_rf_and_colab.ipynb
# so both notebooks share a single source of truth.

CONFIG = {
    # -------------------------------------------------------------------------
    # GEE PROJECT
    # -------------------------------------------------------------------------
    'gee_project': 'data-pipeline-1-464218',

    # -------------------------------------------------------------------------
    # DATE RANGE
    # Dry season (Nov-Feb) gives least cloud cover over Ghana.
    # -------------------------------------------------------------------------
    'start_date':   '2026-3-01',
    'end_date':     '2026-05-29',
    'cloud_thresh': 20,

    # -------------------------------------------------------------------------
    # STUDY AREA  (GeoPackage uploaded to Drive before running)
    # -------------------------------------------------------------------------
    'study_area_file':  'phase2.gpkg',
    'study_area_layer': 'phase2',

    # -------------------------------------------------------------------------
    # EXPORT / FILE NAMES
    # These must match 'features_file' and 'samples_file' in the RF and DL
    # Colab notebooks.  Change the date suffix when targeting a new season.
    # -------------------------------------------------------------------------
    'drive_folder':   'Forest_LULC_Classification',
    'features_file':  'phase2_s1_s2_features_2026.tif',
    'samples_file':   'training_samples.csv',
    'export_crs':     'EPSG:32630',
    'export_scale':   10,


    # -------------------------------------------------------------------------
    # BAND NAMES  (26 bands, same order as the exported feature stack)
    # -------------------------------------------------------------------------
    'band_names': [
        'B2', 'B3', 'B4', 'B5', 'B6', 'B7',
        'B8', 'B8A', 'B11', 'B12',
        'NDVI', 'NDWI', 'NDRE', 'EVI', 'SAVI', 'NBR', 'BSI',
        'VV_mean', 'VH_mean',
        'VV_min',  'VH_min',
        'VV_max',  'VH_max',
        'VV_std',  'VH_std',
        'VH_VV_ratio',
    ],

    # -------------------------------------------------------------------------
    # CLASS MAPPING  (int id -> name, int id -> hex colour)
    # Fill these in to match your training polygons / CSV landcover IDs.
    # They are written to class_mapping.json and read by the RF/DL notebooks.
    # -------------------------------------------------------------------------
    'class_mapping': {
        # Example (uncomment and edit):
        # 0: 'Dense Forest',
        # 1: 'Degraded Forest',
        # 2: 'Water & Swamp',
        # 3: 'Galamsey',
        # 4: 'Bare Land',
        # 5: 'Cocoa Farm',
        # 6: 'Farms',
    },
    'class_colors_map': {
        # Example (uncomment and edit):
        # 0: '#1a6600',
        # 1: '#7aab00',
        # 2: '#00b4cc',
        # 3: '#ff4400',
        # 4: '#d4c48a',
        # 5: '#8b4513',
        # 6: '#ffe066',
    },
}

# Derived palette list for GEE visualisation (strips '#')
CLASS_PALETTE = [v.lstrip('#') for v in CONFIG['class_colors_map'].values()] or [
    '1a6600','7aab00','00b4cc','ff4400','d4c48a','8b4513','ffe066',
]

print('CONFIG loaded.')
print(f'  Drive folder  : {CONFIG["drive_folder"]}')
print(f'  Features file : {CONFIG["features_file"]}')
print(f'  Samples file  : {CONFIG["samples_file"]}')
print(f'  Date range    : {CONFIG["start_date"]} to {CONFIG["end_date"]}')


CONFIG loaded.
  Drive folder  : Forest_LULC_Classification
  Features file : phase2_s1_s2_features_2026.tif
  Samples file  : training_samples.csv
  Date range    : 2026-3-01 to 2026-05-29


In [ ]:
# ============================================================
# AREA OF INTEREST — load study area from Drive
# ============================================================

gpkg_local = drive_download(CONFIG['study_area_file'])
sa_gdf     = gpd.read_file(gpkg_local, layer=CONFIG['study_area_layer']).to_crs('EPSG:4326')
sa_gdf     = sa_gdf.dissolve()
sa_geom    = sa_gdf.geometry.iloc[0]

aoi = ee.Geometry(
    json.loads(sa_gdf.to_json())['features'][0]['geometry']
)



MessageError: User cancelled dfs_ephemeral authorization

In [ ]:
# ============================================================
# SENTINEL-2 — cloud masking, composite, spectral indices
# ============================================================

def mask_s2_clouds(image):
    """Mask clouds using the Scene Classification Layer (SCL band)."""
    scl  = image.select('SCL')
    mask = (
        scl.eq(3)         # cloud shadow
        .Or(scl.eq(8))   # medium probability cloud
        .Or(scl.eq(9))   # high probability cloud
        .Or(scl.eq(10))  # cirrus
        .Not()
    )
    return (
        image.updateMask(mask)
        .divide(10000)
        .copyProperties(image, ['system:time_start'])
    )


def load_sentinel2(aoi, start_date, end_date):
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CONFIG['cloud_thresh']))
        .map(mask_s2_clouds)
    )
    print('S2 images:', collection.size().getInfo())

    composite = collection.median().clip(aoi)
    bands = composite.select(['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12'])

    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndre = composite.normalizedDifference(['B8', 'B5']).rename('NDRE')

    evi = composite.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': composite.select('B8'), 'RED': composite.select('B4'),
         'BLUE': composite.select('B2')}
    ).rename('EVI')

    savi = composite.expression(
        '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
        {'NIR': composite.select('B8'), 'RED': composite.select('B4')}
    ).rename('SAVI')

    nbr = composite.normalizedDifference(['B8', 'B12']).rename('NBR')

    bsi = composite.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {'SWIR': composite.select('B11'), 'RED': composite.select('B4'),
         'NIR':  composite.select('B8'),  'BLUE': composite.select('B2')}
    ).rename('BSI')

    features = (
        bands
        .addBands(ndvi).addBands(ndwi).addBands(ndre)
        .addBands(evi).addBands(savi).addBands(nbr).addBands(bsi)
    )
    print('S2 bands:', features.bandNames().getInfo())
    return composite, features


s2_composite, s2_features = load_sentinel2(aoi, CONFIG['start_date'], CONFIG['end_date'])
print('Sentinel-2 done.')


# ============================================================
# SENTINEL-1 — speckle filter, multi-temporal stats, VH/VV ratio
# ============================================================

def apply_speckle_filter(image):
    """5x5 mean filter to reduce SAR speckle noise."""
    vv = image.select('VV').focal_mean(radius=5, kernelType='square', units='pixels').rename('VV')
    vh = image.select('VH').focal_mean(radius=5, kernelType='square', units='pixels').rename('VH')
    return image.addBands(vv, None, True).addBands(vh, None, True)


def load_sentinel1(aoi, start_date, end_date):
    collection = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .map(apply_speckle_filter)
    )
    print('S1 images:', collection.size().getInfo())

    mean  = collection.select(['VV','VH']).mean().rename(['VV_mean', 'VH_mean'])
    _min  = collection.select(['VV','VH']).min() .rename(['VV_min',  'VH_min' ])
    _max  = collection.select(['VV','VH']).max() .rename(['VV_max',  'VH_max' ])
    std   = (
        collection.select(['VV','VH'])
        .reduce(ee.Reducer.stdDev())
        .rename(['VV_std', 'VH_std'])
    )
    ratio = mean.select('VH_mean').divide(mean.select('VV_mean')).rename('VH_VV_ratio')

    features = (
        mean
        .addBands(_min).addBands(_max)
        .addBands(std).addBands(ratio)
        .clip(aoi)
    )
    print('S1 bands:', features.bandNames().getInfo())
    return features


s1_features = load_sentinel1(aoi, CONFIG['start_date'], CONFIG['end_date'])
print('Sentinel-1 done.')

In [ ]:
# ============================================================
# BUILD COMBINED FEATURE STACK AND VISUALISE
# ============================================================

combined_features = s2_features.addBands(s1_features)
print('Total bands:', combined_features.bandNames().getInfo())



In [ ]:
# ============================================================
# LOAD TRAINING DATA FROM DRIVE
# ============================================================
# Upload one of the following to Drive / Forest_LULC_Classification/
# before running this cell:
#
#   training_dataset.gpkg -- polygon labels (preferred; denser coverage)
#   training_samples.csv  -- point labels with longitude / latitude / landcover columns
#
# The cell tries the GeoPackage first, then falls back to the CSV.

try:
    gpkg_local = drive_download('training_dataset.gpkg')
    gdf = gpd.read_file(gpkg_local).to_crs('EPSG:4326')
    training_data = ee.FeatureCollection([
        ee.Feature(
            ee.Geometry(row.geometry.__geo_interface__),
            {'landcover': int(row['landcover'])}
        )
        for _, row in gdf.iterrows()
    ])
    print(f'Loaded {len(gdf)} training polygons from training_dataset.gpkg')

except FileNotFoundError:
    print('training_dataset.gpkg not found -- trying training_samples.csv')
    csv_local = drive_download(CONFIG['samples_file'])
    df = pd.read_csv(csv_local)
    training_data = ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Point([float(row['longitude']), float(row['latitude'])]),
            {'landcover': int(row['landcover'])}
        )
        for _, row in df.iterrows()
    ])
    print(f'Loaded {len(df)} training points from {CONFIG["samples_file"]}')

print(f'Training features in GEE: {training_data.size().getInfo()}')


In [ ]:
# ============================================================
# SAMPLE FEATURES AT TRAINING LOCATIONS
# ============================================================
# Extracts the 26-band pixel values from the feature stack at every
# training point.  The resulting FeatureCollection is exported as
# training_samples.csv in the next cell.

samples = combined_features.sampleRegions(
    collection = training_data,
    properties = ['landcover'],
    scale      = CONFIG['export_scale'],
    tileScale  = 4,
    geometries = True,   # preserves lon/lat in the exported CSV
)
print('Samples extracted:', samples.size().getInfo())


In [ ]:
# ============================================================
# EXPORT TO GOOGLE DRIVE
# ============================================================
# File names are driven by CONFIG so they automatically match
# what the RF and DL Colab notebooks expect to find on Drive.

FOLDER        = CONFIG['drive_folder']
SCALE         = CONFIG['export_scale']
CRS           = CONFIG['export_crs']
features_stem = CONFIG['features_file'].replace('.tif', '')
samples_stem  = CONFIG['samples_file'].replace('.csv', '')

# 1. Feature stack — primary input for the RF and DL Colab notebooks
task_features = ee.batch.Export.image.toDrive(
    image          = combined_features,
    description    = features_stem,
    folder         = FOLDER,
    fileNamePrefix = features_stem,
    region         = aoi,
    scale          = SCALE,
    crs            = CRS,
    maxPixels      = 1e13,
    fileFormat     = 'GeoTIFF',
)
task_features.start()
print(f'Export started: {CONFIG["features_file"]}')

# 2. Training samples CSV — primary input for the RF and DL Colab notebooks
task_samples = ee.batch.Export.table.toDrive(
    collection     = samples,
    description    = samples_stem,
    folder         = FOLDER,
    fileNamePrefix = samples_stem,
    fileFormat     = 'CSV',
)
task_samples.start()
print(f'Export started: {CONFIG["samples_file"]}')

# 3. Class mapping JSON — consumed by both Colab notebooks for class names / colours
import json as _json
class_mapping_str = _json.dumps(
    {str(k): v for k, v in CONFIG['class_mapping'].items()}, indent=2
)
print('class_mapping.json content:')
print(class_mapping_str)
# Mount Drive and write directly, or upload manually:
# with open(str(Path.home() / 'drive/MyDrive' / FOLDER / 'class_mapping.json'), 'w') as f:
#     f.write(class_mapping_str)

all_tasks = [task_features, task_samples]

for task in all_tasks:
    status = task.status()
    print(f"  {status['description']}: {status['state']}")

print('\nOnce all tasks show COMPLETED, copy the files from')
print(f"Google Drive > {CONFIG['drive_folder']}/")
print('then run the RF or DL classification notebooks.')


In [ ]:
# ============================================================
# MONITOR EXPORT TASKS
# ============================================================
# Re-run this cell at any time to check progress.
# Exports typically take 5-30 minutes depending on AOI size.

